<p style="text-align: center">
<img src="../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>

# Exercise: Write an RRT Path Planner

This exercise builds directly on the C-space collision checker from Exercise 01.
You will implement a **Rapidly-exploring Random Tree (RRT)** path planner for a
differential drive robot navigating a static-obstacle environment.

Your planner will:
1. Grow a tree from the start pose by randomly sampling configurations
2. Use the **`CSpaceChecker`** you already built to reject configurations in collision
3. Use a differential-drive **steering function** (`_steer`) to generate
   kinematically feasible motions between tree nodes
4. Return a sequence of `PlanStep` objects that the robot can execute

You will be working in [`planner.py`](../../packages/planner.py).

**Note: This is a code-only exercise — you do not need a Duckiebot.**

## The RRT Algorithm

RRT grows a tree rooted at the start pose. At each iteration it:

1. **Samples** a random configuration q_rand in the space (with occasional **goal bias**)
2. **Finds** the nearest node in the current tree to q_rand
3. **Steers** from q_near toward q_rand, producing a kinematically feasible path
4. **Checks** whether that path is collision-free using the C-space checker
5. **Adds** the new node (and the path to it) to the tree if collision-free
6. **Terminates** early if the new node is within tolerance of the goal

```
tree <- {q_start}

for _ in range(max_iter):
    if uniform() < goal_bias:
        q_rand <- q_target                      # goal bias
    else:
        q_rand <- sample uniformly in bounds    # exploration

    q_near        <- nearest_node(tree, q_rand)
    steps, q_new  <- steer(q_near, q_rand)

    if path_collision_free(q_near -> q_new):
        tree.add(q_new, parent=q_near, steps=steps)
        if reached(q_new, q_target):
            return extract_path(tree, q_new)   # success

return INFEASIBLE
```

**Why goal bias?**  
Pure random sampling rarely lands near the goal. With probability `goal_bias`
(typically 0.05–0.15) you sample the goal directly, which steers the tree
toward it and dramatically reduces the expected number of iterations.

**Path extraction**  
Each node stores a pointer to its parent and the `PlanStep` list used to reach it.
To recover the full plan, walk from the goal node back to the root following parent
pointers, collect the edge steps, then reverse and concatenate.

## Steering Function: One Primitive at a Time

A differential drive robot obeys one strict kinematic rule:
**it cannot rotate and translate simultaneously.**
At any moment the robot is doing exactly one of three things:

| Primitive | `velocity_x_m_s` | `angular_velocity_deg_s` |
|-----------|-----------------|-------------------------|
| Drive straight | > 0 | = 0 |
| Turn left in place | = 0 | > 0 |
| Turn right in place | = 0 | < 0 |

Each RRT extension adds **one** such primitive to the tree.
The steering function decides which one:

```
def _steer(q_near, q_rand, max_v, max_w):
    dx, dy = q_rand.x - q_near.x, q_rand.y - q_near.y
    dist   = sqrt(dx**2 + dy**2)

    if dist > threshold:
        heading_error = angle from q_near.theta toward (dx, dy)
        if |heading_error| > TURN_THRESHOLD:
            # face q_rand first
            return turn_step(heading_error, max_w)
        else:
            # aligned — drive straight (cap at STEP_DIST_M)
            return straight_step(min(dist, STEP_DIST_M), max_v)
    else:
        # same position — turn to match q_rand's orientation
        return turn_step(q_rand.theta - q_near.theta, max_w)
```

Because each primitive has **either** zero linear velocity **or** zero angular velocity,
the kinematic constraint is satisfied by construction.
The resulting path looks like an alternating sequence of rotations (robot stays in place)
and straight segments (robot doesn't rotate).

Key helper: `normalize_angle` (from `utils.py`) wraps an angle to [−π, π],
which is essential for computing the shortest-direction turn.

## Data Structures

See [`collision_protocol.py`](../../packages/collision_protocol.py) for full definitions.  
The planner receives a `PlanningSetup` and a `PlanningQuery` and returns a `PlanningResult`.

```python
@dataclass
class PlanningSetup(MapDefinition):
    bounds: Rectangle               # allowed area
    max_linear_velocity_m_s: float
    min_linear_velocity_m_s: float
    max_angular_velocity_deg_s: float
    max_curvature: float            # math.inf for a differential drive robot
    tolerance_xy_m: float           # goal-reached position tolerance
    tolerance_theta_deg: float      # goal-reached orientation tolerance

@dataclass
class PlanningQuery:
    start: FriendlyPose
    target: FriendlyPose

@dataclass
class PlanningResult:
    feasible: bool
    plan: Optional[List[PlanStep]]

@dataclass
class PlanStep:
    duration: float
    velocity_x_m_s: float
    angular_velocity_deg_s: float
```

`MapDefinition` (parent of `PlanningSetup`) contains:
- `environment: List[PlacedPrimitive]` — static obstacles
- `body: List[PlacedPrimitive]` — robot body primitives (centred at origin)

## Tips for Implementation

**Start with an empty environment.**  
Set `n_circles=0, n_rectangles=0`. Confirm the planner can connect start to goal
before adding obstacles.

**_steer produces one PlanStep inside a list.**  
Return `([step], q_new)` — a list with a single element. This keeps the interface
consistent with `edge_steps` (which stores lists of PlanStep).

**Check every pose along a path, not just the endpoint.**  
Use `more_granular(steps, dt)` (e.g., `dt=0.05`) from `utils.py` to densify
the trajectory, then `simulate(start, fine_steps).poses` to get all poses,
then `checker.check(pose)` on each one. During a turn step the (x, y) position
is constant but orientation sweeps — the C-space checker handles this correctly.

**Nearest-neighbour metric.**  
Use mainly position distance (since orientation is cheap to fix with a turn step):
```python
sqrt(dx**2 + dy**2) + 0.1 * abs(normalize_angle(deg2rad(p.theta - q.theta)))
```

**Tree data structures.**  Three parallel lists:
```python
nodes:      List[FriendlyPose]             # one pose per node
parents:    List[Optional[int]]            # parent index (None for root)
edge_steps: List[Optional[List[PlanStep]]] # single-step list reaching this node
```

**Tuning.**  
The provided `_STEP_DIST_M=0.12` takes small steps so this same planner also
clears the tight tile gaps in the duckiematrix map; that finer step needs a
larger budget — `max_iter=40000` (`goal_bias=0.10`) solves this 3 m × 3 m room.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
from matplotlib import pyplot as plt

In [ ]:
import math
import numpy as np
from numpy.random import RandomState

from collision_protocol import FriendlyPose, PlanningSetup, PlanningQuery, Rectangle
from collision_checker import CSpaceChecker
from make_environments_planning import (
    PlanningParams, FixedPlanningParams, FixedBody, RandomEnv, SamplingEnvInfo,
)
from planner import plan
from utils import simulate
from collision_drawing import plot_geometry
from se2_utils import SE2, pose_from_friendly
from make_environments import COLOR_BG

# -- Environment ----------------------------------------------------------
bounds = Rectangle(xmin=0.0, xmax=3.0, ymin=0.0, ymax=3.0)
sei = SamplingEnvInfo(
    n_circles=4,
    n_rectangles=4,
    min_distance=0.2,
    interval_radius=(0.15, 0.35),
    interval_sides=(0.10, 0.25),
)
rs = RandomState(7)
env_info = RandomEnv(bounds, sei).sample(rs)
body     = FixedBody().sample(rs)

# -- Planning parameters --------------------------------------------------
pp = PlanningParams(
    tolerance_xy_m=0.1,
    tolerance_theta_deg=20.0,
    min_linear_velocity_m_s=-0.3,
    max_linear_velocity_m_s=0.4,
    max_curvature=math.inf,
    max_angular_velocity_deg_s=30.0,
)
ps = PlanningSetup(
    body=body,
    environment=env_info.obstacles,
    bounds=bounds,
    max_linear_velocity_m_s=pp.max_linear_velocity_m_s,
    min_linear_velocity_m_s=pp.min_linear_velocity_m_s,
    max_angular_velocity_deg_s=pp.max_angular_velocity_deg_s,
    max_curvature=pp.max_curvature,
    tolerance_xy_m=pp.tolerance_xy_m,
    tolerance_theta_deg=pp.tolerance_theta_deg,
)

# -- Start / target -------------------------------------------------------
start  = FriendlyPose(0.4, 0.4,   0.0)
target = FriendlyPose(2.5, 2.5,  90.0)

# -- Always plot environment (so you can verify feasibility visually) -----
checker = CSpaceChecker(ps.environment, ps.body)
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_aspect("equal")
ax.axis((bounds.xmin, bounds.xmax, bounds.ymin, bounds.ymax))
ax.set_facecolor(COLOR_BG)
plot_geometry(ax, SE2.identity(), ps.environment, None, 0)
plot_geometry(ax, pose_from_friendly(start),  ps.body, "green", 8)
plot_geometry(ax, pose_from_friendly(target), ps.body, "red",   8)
ax.plot(start.x,  start.y,  "go", markersize=10, zorder=10, label="start")
ax.plot(target.x, target.y, "r*", markersize=15, zorder=10, label="target")

start_ok  = not checker.check(start)
target_ok = not checker.check(target)
print(f"Start  collision-free: {start_ok}")
print(f"Target collision-free: {target_ok}")

# -- Run planner ----------------------------------------------------------
print("\nPre-computing C-space slices and running RRT...")
# smaller motion-primitive steps (tuned for the duckiematrix tile gaps) need a bigger RRT budget
result = plan(ps, PlanningQuery(start=start, target=target), max_iter=40000)
print(f"Feasible: {result.feasible}")

if result.feasible:
    sim = simulate(start, result.plan)
    total_time = sum(s.duration for s in result.plan)
    print(f"Plan: {len(result.plan)} steps, {total_time:.1f} s total")

    xs = [p.x for p in sim.poses]
    ys = [p.y for p in sim.poses]
    ax.plot(xs, ys, "b-", linewidth=2, zorder=5, label="path")

    # Draw ghosts only after drive steps (not after in-place turns),
    # so every ghost is at a distinct position and faces its travel direction.
    drive_indices = [i + 1 for i, s in enumerate(result.plan)
                     if s.velocity_x_m_s > 0.01]
    n_ghosts = 6
    if len(drive_indices) > n_ghosts:
        sub = max(1, len(drive_indices) // n_ghosts)
        drive_indices = drive_indices[::sub]
    for i in drive_indices:
        plot_geometry(ax, pose_from_friendly(sim.poses[i]), ps.body, "gray", 6)

    ax.set_title(f"RRT path -- {len(result.plan)} steps, {total_time:.1f} s")
else:
    ax.set_title("No path found (infeasible or max_iter reached)")

ax.legend()
plt.show()


## Watch the Tree Grow

RRT is much easier to understand when you can *see* it explore. The planner
accepts an optional `on_extend(parent, child)` callback that fires every time a
new node is added to the tree. The cell below passes a callback that records
each `(parent -> child)` edge, then animates the tree filling the free space
until it snaps onto the goal (the final blue path).

This is purely for visualization -- your `plan()` never needs `on_extend` to
work. To enable the animation, add these two lines right after you append a
node in your RRT loop:

```python
if on_extend is not None:
    on_extend(nodes[nearest_idx], q_new)
```


In [ ]:
# ---- Animate the RRT as it grows -----------------------------------------
# Requires your plan() to call on_extend(parent, child) each time it adds a node.
import numpy as np
from matplotlib.animation import FuncAnimation
from matplotlib.collections import LineCollection
from IPython.display import HTML

edges = []
res_viz = plan(
    ps,
    PlanningQuery(start=start, target=target),
    max_iter=40000,
    on_extend=lambda parent, child: edges.append([(parent.x, parent.y), (child.x, child.y)]),
)
print(f"Recorded {len(edges)} edges; feasible={res_viz.feasible}")
if not edges:
    print("No edges recorded - call on_extend(parent, child) inside your plan() to enable the animation.")

# One frame every few hundred edges (RRT returns as soon as it reaches the goal).
n_frames = min(100, max(1, len(edges)))
frame_sizes = np.linspace(1, max(1, len(edges)), n_frames, dtype=int)

figA, axA = plt.subplots(figsize=(6.5, 6.5))
axA.set_aspect("equal")
axA.axis((bounds.xmin, bounds.xmax, bounds.ymin, bounds.ymax))
axA.set_facecolor(COLOR_BG)
plot_geometry(axA, SE2.identity(), ps.environment, None, 0)
axA.plot(start.x, start.y, "go", markersize=10, zorder=5, label="start")
axA.plot(target.x, target.y, "r*", markersize=15, zorder=5, label="goal")
axA.legend(loc="upper left")

tree_lc = LineCollection([], colors="steelblue", linewidths=0.4, alpha=0.6, zorder=2)
axA.add_collection(tree_lc)
path_line, = axA.plot([], [], "b-", linewidth=2.5, zorder=6)

if res_viz.feasible:
    sim_viz = simulate(start, res_viz.plan)
    path_x = [p.x for p in sim_viz.poses]
    path_y = [p.y for p in sim_viz.poses]

def _update(k):
    tree_lc.set_segments(edges[: frame_sizes[k]])
    axA.set_title(f"RRT growth: {frame_sizes[k]} nodes")
    if k == len(frame_sizes) - 1 and res_viz.feasible:
        path_line.set_data(path_x, path_y)
    return tree_lc, path_line

anim = FuncAnimation(figA, _update, frames=len(frame_sizes), interval=80, blit=False)
plt.close(figA)  # show only the animation, not the static first frame
HTML(anim.to_jshtml())
